This EDA uses corrected calendar-day labels and disjoint as-of cohorts under `artifacts/corrected/tables`. Final refit has its own mature-label pool; see the manifest.


# Task 2 — Notebook 4: Exploratory Data Analysis

This notebook explores the **training split only**. The validation and test sets are not used here, so their later observations remain untouched until model evaluation.

The purpose of this EDA is to understand data quality, class imbalance, distributions, and simple relationships that can guide feature engineering.

**Input:** `artifacts/tables/03_train.parquet`

**Outputs:** charts under `artifacts/charts/` and a short report at `artifacts/reports/04_findings.md`


## 1. Imports and paths


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

import sys

src_dir = Path.cwd().resolve()
while src_dir != src_dir.parent and not (src_dir / "Task2").is_dir():
    src_dir = src_dir.parent
src_dir = src_dir / "Task2" / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from config import find_project_root

PROJECT_ROOT = find_project_root()
TASK2_ROOT = PROJECT_ROOT / "Task2"
ARTIFACTS_DIR = TASK2_ROOT / "artifacts"
TABLES_DIR = ARTIFACTS_DIR / "corrected" / "tables"
REPORTS_DIR = ARTIFACTS_DIR / "corrected" / "reports"
CHARTS_DIR = ARTIFACTS_DIR / "corrected" / "charts"

for folder in [REPORTS_DIR, CHARTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

input_path = TABLES_DIR / "03_train.parquet"
if not input_path.exists():
    raise FileNotFoundError(f"Training artifact not found: {input_path}")


## 2. Load the training data

The target column is `is_late`. Columns that describe events after delivery, such as the actual delivery timestamp, must not be used as predictive features later because they would leak the answer.


In [ ]:
train = pd.read_parquet(input_path)

train["order_purchase_timestamp"] = pd.to_datetime(
    train["order_purchase_timestamp"], errors="coerce"
)

assert "is_late" in train.columns
assert train["order_id"].is_unique

print(f"Rows: {len(train):,}")
print(f"Columns: {len(train.columns)}")
print(f"Late-delivery rate: {train['is_late'].mean() * 100:.2f}%")
display(train.head())


## 3. Data quality overview


In [ ]:
quality = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "missing_count": train.isna().sum(),
    "missing_percentage": (train.isna().mean() * 100).round(2),
    "unique_values": train.nunique(dropna=True),
}).sort_values("missing_percentage", ascending=False)

display(quality)
print(f"Duplicate rows: {train.duplicated().sum():,}")


In [ ]:
missing = quality[quality["missing_count"] > 0].head(10)

if len(missing) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(
        data=missing.reset_index(names="column"),
        x="missing_percentage",
        y="column",
        color="#f59e0b",
        ax=ax,
    )
    ax.set_title("Top columns with missing values")
    ax.set_xlabel("Missing values (%)")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()
    fig.savefig(CHARTS_DIR / "04_missing_values.png", dpi=150, bbox_inches="tight")
else:
    print("No missing values were found.")


## 4. Target distribution

The late-delivery class is expected to be smaller than the on-time class. This imbalance must be considered during model training and evaluation.


In [ ]:
target_counts = train["is_late"].value_counts().sort_index()
target_table = pd.DataFrame({
    "label": ["on_time", "late"],
    "count": [target_counts.get(0, 0), target_counts.get(1, 0)],
})
target_table["percentage"] = (target_table["count"] / len(train) * 100).round(2)
display(target_table)

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=target_table, x="label", y="count", color="#2563eb", ax=ax)
ax.set_title("Training target distribution")
ax.set_xlabel("Delivery label")
ax.set_ylabel("Number of orders")
for container in ax.containers:
    ax.bar_label(container, fmt="%d")
plt.tight_layout()
plt.show()
fig.savefig(CHARTS_DIR / "04_target_distribution.png", dpi=150, bbox_inches="tight")


## 5. Numerical feature distributions

The selected variables describe the order before delivery: price, freight, item counts, payment information, and product information. Skewed variables may need transformations or robust scaling in a later notebook.


In [ ]:
candidate_numeric = [
    "total_price", "total_freight", "total_payment_value",
    "num_items", "num_distinct_products", "num_sellers",
    "num_product_categories", "avg_product_weight_g",
    "avg_product_photos_qty", "max_payment_installments",
    "avg_review_score",
]
numeric_columns = [column for column in candidate_numeric if column in train.columns]

numeric_summary = train[numeric_columns].describe().T
numeric_summary["missing"] = train[numeric_columns].isna().sum()
display(numeric_summary)


In [ ]:
plot_columns = [
    column for column in ["total_price", "total_freight", "num_items", "avg_review_score"]
    if column in train.columns
]

if plot_columns:
    train[plot_columns].hist(figsize=(10, 7), bins=30, edgecolor="white")
    plt.suptitle("Selected numerical feature distributions", y=1.02)
    plt.tight_layout()
    plt.show()
    plt.savefig(CHARTS_DIR / "04_numeric_distributions.png", dpi=150, bbox_inches="tight")


## 6. Late-delivery rate by customer state

This is a simple group comparison, not a causal conclusion. It can show whether geographic information may be useful for a later model.


In [ ]:
if "customer_state" in train.columns:
    state_summary = (
        train.groupby("customer_state", as_index=False)
        .agg(orders=("order_id", "count"), late_rate=("is_late", "mean"))
    )
    state_summary["late_rate_percentage"] = (state_summary["late_rate"] * 100).round(2)
    state_summary = state_summary.sort_values("late_rate", ascending=False)
    display(state_summary.head(10))

    fig, ax = plt.subplots(figsize=(9, 5))
    sns.barplot(
        data=state_summary,
        x="customer_state",
        y="late_rate_percentage",
        color="#0f766e",
        ax=ax,
    )
    ax.set_title("Late-delivery rate by customer state")
    ax.set_xlabel("Customer state")
    ax.set_ylabel("Late orders (%)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    fig.savefig(CHARTS_DIR / "04_late_rate_by_state.png", dpi=150, bbox_inches="tight")


## 7. Late-delivery rate over time

The monthly view helps identify changes in delivery performance across the purchase period.


In [ ]:
train["purchase_month"] = train["order_purchase_timestamp"].dt.to_period("M").astype(str)
monthly_summary = (
    train.groupby("purchase_month", as_index=False)
    .agg(orders=("order_id", "count"), late_rate=("is_late", "mean"))
)
monthly_summary["late_rate_percentage"] = (monthly_summary["late_rate"] * 100).round(2)
display(monthly_summary)

fig, ax = plt.subplots(figsize=(12, 4))
sns.lineplot(
    data=monthly_summary,
    x="purchase_month",
    y="late_rate_percentage",
    marker="o",
    color="#dc2626",
    ax=ax,
)
ax.set_title("Monthly late-delivery rate")
ax.set_xlabel("Purchase month")
ax.set_ylabel("Late orders (%)")
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()
fig.savefig(CHARTS_DIR / "04_monthly_late_rate.png", dpi=150, bbox_inches="tight")


## 8. Record findings for the next notebook


In [ ]:
leakage_columns = [
    column for column in [
        "order_delivered_customer_date",
        "order_delivered_carrier_date",
        "delivery_days",
    ]
    if column in train.columns
]

findings = f'''# Task 2 — Notebook 4 EDA Findings

## Dataset

- Training rows: {len(train):,}
- Columns: {len(train.columns):,}
- Late-delivery rate: {train['is_late'].mean() * 100:.2f}%
- Duplicate rows: {int(train.duplicated().sum()):,}

## Data quality

The table above reports data types, missing counts, missing percentages, and unique-value counts. Missing values should be handled inside the modeling pipeline rather than filled using information from validation or test data.

## Modeling considerations

- The target is imbalanced, so accuracy alone is not sufficient for evaluation.
- Price, freight, payment, item-count, product, and geographic variables are candidates for feature engineering.
- Skewed numeric variables may benefit from robust preprocessing or transformations.
- The following post-delivery columns must be excluded from predictive features to avoid leakage: {', '.join(leakage_columns) if leakage_columns else 'none found in this table'}.
- This EDA used the training split only.
'''

report_path = REPORTS_DIR / "04_findings.md"
report_path.write_text(findings, encoding="utf-8")
print(f"Saved EDA findings to: {report_path}")


## 9. Final checks


In [ ]:
assert len(train) > 0
assert train["is_late"].isin([0, 1]).all()
assert (REPORTS_DIR / "04_findings.md").exists()
assert (CHARTS_DIR / "04_target_distribution.png").exists()
print("ALL NOTEBOOK 4 CHECKS PASSED")
